# Funnel & Behavioural Analysis

The funnel is where you find the real conversion problems. View → Click → Add to Cart → Purchase — at each step you're losing people, and the question is *where* you're losing the most and *why*.

The behavioural analysis layer (device types, traffic sources, time patterns) adds context to those drop-off numbers. Losing 60% between Click and Add to Cart on mobile looks different if mobile converts well on other traffic sources — it might point to a specific campaign landing page issue rather than a general mobile UX problem.


In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.load import get_connection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

con = get_connection()
events = con.execute("SELECT * FROM events").df()
funnel = con.execute("SELECT * FROM mart_funnel").df()
events['timestamp'] = pd.to_datetime(events['timestamp'])
events['hour'] = events['timestamp'].dt.hour
events['day_of_week'] = events['timestamp'].dt.day_name()


## The overall funnel

The headline numbers. The view-to-purchase rate is the metric most people care about, but the stage-by-stage drop-off is where you find the actionable insight. A huge drop between view and click suggests the product pages aren't compelling enough. A big drop between add-to-cart and purchase is a classic checkout abandonment problem.


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db','#2ecc71','#f1c40f','#e74c3c']
bars = ax.barh(funnel['stage'][::-1], funnel['count'][::-1], color=colors, edgecolor='white')
for bar, (_, row) in zip(bars, funnel[::-1].iterrows()):
    ax.text(bar.get_width()+10000, bar.get_y()+bar.get_height()/2,
            f"{row['count']:,}  ({row['conversion_from_top']*100:.1f}%)",
            va='center', fontsize=9)
ax.set_title('Conversion Funnel — All Events')
ax.set_xlabel('Event Count'); plt.tight_layout(); plt.show()
print(funnel.to_string(index=False))


## Funnel by device type

Mobile vs desktop conversion is one of the most common things teams argue about. The data here should settle it. If mobile converts significantly worse at a specific stage, that's where the UX investment should go. If mobile actually converts better (which happens more than people expect on mobile-first product types), that should inform where you put your ad spend.


In [ ]:

funnel_stages = ['view','click','add_to_cart','purchase']
ev_dev = events[events['event_type'].isin(funnel_stages)]
dev_funnel = ev_dev.groupby(['device_type','event_type'])['event_id'].count().unstack(fill_value=0)
dev_funnel = dev_funnel[[s for s in funnel_stages if s in dev_funnel.columns]]
dev_funnel_pct = dev_funnel.div(dev_funnel['view'], axis=0).round(4)
print(dev_funnel_pct)
dev_funnel_pct.T.plot(kind='bar', figsize=(9, 5))
plt.title('Funnel Conversion by Device Type'); plt.ylabel('Rate (relative to views)')
plt.xticks(rotation=30, ha='right'); plt.legend(title='Device'); plt.tight_layout(); plt.show()


## Traffic source conversion rate

Not all traffic is equal. A user who clicked a targeted email campaign has very different intent than someone who landed via a generic social ad. This breakdown shows which sources actually deliver converting traffic — which should directly inform where budget gets allocated.

High bounce rate + low conversion on a source that's receiving significant investment is a strong signal to either fix the landing experience for that source or reallocate the budget.


In [ ]:

src = events.groupby('traffic_source').agg(
    views=('event_type', lambda x: (x=='view').sum()),
    carts=('event_type', lambda x: (x=='add_to_cart').sum()),
    purchases=('event_type', lambda x: (x=='purchase').sum()),
    bounces=('event_type', lambda x: (x=='bounce').sum()),
).assign(
    conversion_rate=lambda d: (d['purchases'] / d['views']).round(4),
    bounce_rate=lambda d: (d['bounces'] / (d['views']+d['bounces'])).round(4)
).sort_values('conversion_rate', ascending=False)
print(src)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
src['conversion_rate'].plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Conversion Rate by Traffic Source'); plt.sca(axes[0]); plt.xticks(rotation=30, ha='right')
src['bounce_rate'].plot(kind='bar', ax=axes[1], color='crimson', edgecolor='white')
axes[1].set_title('Bounce Rate by Traffic Source'); plt.sca(axes[1]); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()


## When are users most active? Day × hour heatmap

This heatmap shows event volume by day and hour. The practical use cases: scheduling email sends, timing push notifications, planning flash sales, and deciding when to run paid campaigns (bidding up during peak hours can make sense if conversion holds).

Dark spots on certain days/hours might point to server issues or just genuine low-traffic periods — worth cross-referencing with conversion rate to distinguish the two.


In [ ]:

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = events.groupby(['day_of_week','hour'])['event_id'].count().unstack(fill_value=0)
pivot = pivot.reindex(dow_order)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.3)
ax.set_title('Event Volume — Day of Week × Hour of Day'); ax.set_xlabel('Hour'); ax.set_ylabel('')
plt.tight_layout(); plt.show()


## Session duration by page category

Longer sessions aren't always better — someone spending 10 minutes on a checkout page is probably confused, not engaged. But on product detail pages (PDPs), longer sessions often correlate with purchase intent. Context matters here, which is why breaking it down by page category is more useful than a single average.


In [ ]:

sess = events.groupby('page_category')['session_duration_sec'].agg(['mean','median','count']).round(1).sort_values('mean', ascending=False)
print(sess)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sess.index, sess['mean'], color='steelblue', edgecolor='white', label='Mean')
ax.bar(sess.index, sess['median'], color='coral', edgecolor='white', alpha=0.7, label='Median')
ax.set_title('Session Duration by Page Category'); ax.set_ylabel('Seconds')
ax.legend(); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print("\nNote: mean > median on most categories, confirming right-skewed session times — a few long sessions are pulling the average up.")
